<a href="https://colab.research.google.com/github/SantiDrelewicz/prode/blob/main/notebooks/EvaluacionProde.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import LeaveOneGroupOut, RandomizedSearchCV
from sklearn.metrics import mean_poisson_deviance

In [ ]:
df = pd.read_parquet("https://github.com/SantiDrelewicz/prode/raw/refs/heads/main/data/partidos_desagregados_lv.parquet")

In [ ]:
df

,match_id,league,date,season,team,opponent,gf,ga,xg,xga,...,eds_opp,roll_xg_diff_6,roll_goal_diff_6,diff_elo,desbalance_off,desbalance_def,desbalance_eff_off,desbalance_eff_def,target_goles,is_home
0,5447,Bundesliga,2014-08-22 19:30:00,2014,Bayern Munich,Wolfsburg,2,1,2.570120,1.198420,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,1
1,5448,Bundesliga,2014-08-23 14:30:00,2014,Hoffenheim,Augsburg,2,0,1.528730,0.280777,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,1
2,5449,Bundesliga,2014-08-23 14:30:00,2014,Hannover 96,Schalke 04,2,1,1.179790,0.956665,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,1
3,5450,Bundesliga,2014-08-23 14:30:00,2014,Hertha Berlin,Werder Bremen,2,2,1.755850,1.194530,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,1
4,5451,Bundesliga,2014-08-23 14:30:00,2014,Eintracht Frankfurt,Freiburg,1,0,1.753310,1.380840,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48933,30215,Serie_A,2026-05-24 18:45:00,2025,Cremonese,Como,1,4,0.977192,2.691100,...,-0.148924,0.165426,0.146699,198.967108,0.779578,-0.372352,-0.236178,-0.266193,4,0
48934,30218,Serie_A,2026-05-24 18:45:00,2025,Lecce,Genoa,1,0,2.213870,0.782331,...,-0.333664,-0.697802,-0.813132,51.965347,-0.352311,0.063970,-0.447331,-0.305904,0,0
48935,30219,Serie_A,2026-05-24 18:45:00,2025,AC Milan,Cagliari,1,2,1.759530,3.436240,...,-0.179252,0.281474,-0.510004,-193.538736,0.439072,-0.850727,-0.287152,-0.361770,2,0
48936,30222,Serie_A,2026-05-24 18:45:00,2025,Torino,Juventus,2,2,0.950066,2.534270,...,-0.021361,0.492407,0.771617,172.398939,1.229653,-1.293839,-0.288429,-0.218932,2,0


In [ ]:
# ==============================================================================
# 2. DEFINICIÓN DE LOS MODELOS
# ==============================================================================

FEATURE_SETS = {
    "1_Baseline_Localia": [
        'is_home'
    ],
    '2_Elo': [
        'diff_elo',
        'is_home'
    ],

    '3_Prodegy_Puro': [
        'xos_team',
        'xds_opp',
        'is_home'
    ],

    '4_Prodegy_Con_Eff': [
        'xos_team',
        'xds_opp',
        'eos_team',
        'is_home'
    ],

    '5_Prodegy_Desbalance': [
        'desbalance_off',
        'is_home'
    ],

    '6_Desbalances': [
        'diff_elo',
        'xos_team',
        'desbalance_off',
        'desbalance_eff_off',
        'is_home'
    ],

    '7_Full_System': [
        'diff_elo',
        'elo_team',
        'elo_opp',
        'xos_team',
        'xds_team',
        'xos_opp',
        'xds_opp',
        'desbalance_off',
        'desbalance_def',
        'eos_team',
        'eds_team',
        'eos_opp',
        'eds_opp',
        'desbalance_eff_off',
        'desbalance_eff_def',
        'is_home'
    ]
}

# ==============================================================================
# 3. HIPERPARÁMETROS PARA LOS MODELOS COMPLEJOS
# ==============================================================================
PARAM_DICT = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_iter': [100, 300],
    'max_depth': [3, 5, 7, None],
    'l2_regularization': [0.0, 0.1, 1.0]
}

N_ITER_RANDOM_SEARCH = 10

# ==============================================================================
# 4. CONFIGURACIÓN GENERAL
# ==============================================================================
MIN_SEASON, MAX_SEASON = 2019, 2025

# Cantidad mínima de temporadas anteriores necesarias
# para hacer CV temporal dentro del entrenamiento.
MIN_TRAIN_SEASONS = 2

RANDOM_STATE = 42

LOGO = LeaveOneGroupOut()

In [ ]:
def bootstrap(y_pred_oof, y_true, B):
  estimaciones = []
  for b in range(B):
    muestra_boot = np.random.choice(range(len(y_pred_oof)), size=len(y_pred_oof), replace=True)
    y_pred_boot = y_pred_oof[muestra_boot]
    y_true_boot = y_true[muestra_boot]
    estimaciones.append(mean_poisson_deviance(y_true_boot, y_pred_boot))

  return np.mean(estimaciones), np.std(estimaciones, ddof=1)



In [ ]:
"""Nested CV temporal por temporada para el proyecto de predicción de fútbol.

Este módulo NO preprocesa el dataframe: se supone que `df` ya tiene
las features listas para entrenar. Sólo selecciona temporadas para los splits.

- INTRA: entrenar con temporadas anteriores de la liga objetivo.
- GLOBAL: entrenar con temporadas anteriores de todas las ligas.
- Tanto la validación interna como la externa evalúan la liga objetivo.
- Los folds internos son expansivos y ajustan los hiperparámetros.
- Los folds externos estiman el rendimiento del entrenamiento completo.
"""

def folds_temporales(df_train, liga_objetivo, min_train_seasons=1):
    """Devuelve splits (índices relativos a df_train) para la CV interna.

    Un fold valida una temporada completa de `liga_objetivo` y entrena
    con TODAS las filas de temporadas anteriores presentes en df_train.
    Por eso sirve tanto para INTRA como para GLOBAL.
    """
    temporadas = np.sort(
        df_train.loc[df_train["league"].eq(liga_objetivo), "season"].unique()
    )
    grupos = df_train["season"].to_numpy()
    ligas = df_train["league"].to_numpy()
    splits = []

    for temporada_val in temporadas[min_train_seasons:]:
        idx_train = np.flatnonzero(grupos < temporada_val)
        idx_val = np.flatnonzero((grupos == temporada_val) & (ligas == liga_objetivo))
        if len(idx_train) and len(idx_val):
            splits.append((idx_train, idx_val))

    if not splits:
        raise ValueError(
            f"No se pudieron generar folds internos para {liga_objetivo}. "
            "Revisá la cantidad de temporadas y min_train_seasons."
        )
    return splits


def entrenar_modelo(
    df_train,
    columnas_features,
    liga_objetivo,
    param_dict,
    *,
    n_iter=10,
    min_train_seasons_inner=1,
    random_state=42,
    n_jobs=-1,
):
    """Ajusta un estimador sin observar los datos del fold externo.

    Conserva la regla del notebook: hasta dos features => PoissonRegressor;
    más de dos => HistGradientBoostingRegressor + búsqueda temporal.
    """
    X_train = df_train[columnas_features]
    y_train = df_train["target_goles"]

    if len(columnas_features) <= 2:
        modelo = PoissonRegressor(max_iter=1000)
        modelo.fit(X_train, y_train)
        return modelo, {"modelo": "PoissonRegressor"}, np.nan, 0

    cv_interna = folds_temporales(
        df_train, liga_objetivo, min_train_seasons=min_train_seasons_inner
    )
    buscador = RandomizedSearchCV(
        estimator=HistGradientBoostingRegressor(
            loss="poisson", early_stopping=False, random_state=random_state
        ),
        param_distributions=param_dict,
        n_iter=n_iter,
        cv=cv_interna,
        scoring="neg_mean_poisson_deviance",
        random_state=random_state,
        n_jobs=n_jobs,
        refit=True,
        error_score="raise",
    )
    buscador.fit(X_train, y_train)
    return (
        buscador.best_estimator_,
        buscador.best_params_,
        -float(buscador.best_score_),
        len(cv_interna),
    )


def temporal_nested_cv(
    df,
    feature_sets,
    param_dict,
    *,
    min_season=None,
    max_season=None,
    test_season=None,
    min_train_seasons=2,
    min_train_seasons_inner=1,
    n_iter=10,
    random_state=42,
    n_jobs=-1,
    verbose=False,
):
    """Compara modelos mediante Nested CV temporal sin preprocesar el dataset.

    `test_season`, si se especifica, se excluye de TODA la CV y queda
    reservado para evaluar la elección final en un paso posterior.

    Devuelve: detalle, por_liga, resumen, comparacion y test.
    """
    if min_train_seasons < min_train_seasons_inner + 1:
        raise ValueError(
            "min_train_seasons debe ser al menos min_train_seasons_inner + 1 "
            "para que el primer fold externo tenga un fold interno."
        )
    if min_train_seasons_inner < 1:
        raise ValueError("min_train_seasons_inner debe ser >= 1.")
    if not feature_sets:
        raise ValueError("feature_sets está vacío.")

    columnas = {"league", "season", "target_goles"}
    columnas.update(c for features in feature_sets.values() for c in features)
    faltantes = columnas - set(df.columns)
    if faltantes:
        raise ValueError(f"Columnas faltantes en df: {sorted(faltantes)}")
    if df[["league", "season", "target_goles"]].isna().any().any():
        raise ValueError("Hay nulos en league, season o target_goles.")
    if (df["target_goles"] < 0).any():
        raise ValueError("target_goles debe ser >= 0 para los modelos Poisson.")

    # Único filtrado: fechas/temporadas elegidas, no se transforma ninguna feature.
    datos = df
    if min_season is not None:
        datos = datos.loc[datos["season"] >= min_season]
    if max_season is not None:
        datos = datos.loc[datos["season"] <= max_season]

    if test_season is not None:
        test = datos.loc[datos["season"] == test_season]
        if test.empty:
            raise ValueError(f"No hay datos de la temporada test {test_season}.")
        desarrollo = datos.loc[datos["season"] < test_season]
    else:
        test = datos.iloc[0:0]
        desarrollo = datos

    if desarrollo.empty:
        raise ValueError("No hay datos de desarrollo.")

    resultados = []
    for liga in sorted(desarrollo["league"].unique()):
        df_liga = desarrollo.loc[desarrollo["league"] == liga]
        temporadas = np.sort(df_liga["season"].unique())

        for temporada_val in temporadas[min_train_seasons:]:
            # La temporada externa queda COMPLETAMENTE fuera de los entrenamientos.
            df_val = df_liga.loc[df_liga["season"] == temporada_val]
            df_historico = desarrollo.loc[desarrollo["season"] < temporada_val]

            for estrategia in ("INTRA", "GLOBAL"):
                df_train = (
                    df_historico.loc[df_historico["league"] == liga]
                    if estrategia == "INTRA"
                    else df_historico
                )
                for nombre_modelo, features in feature_sets.items():
                    if verbose:
                        print(f"{liga} | {temporada_val} | {estrategia} | {nombre_modelo}")
                    modelo, params, score_int, n_folds_int = entrenar_modelo(
                        df_train,
                        features,
                        liga,
                        param_dict,
                        n_iter=n_iter,
                        min_train_seasons_inner=min_train_seasons_inner,
                        random_state=random_state,
                        n_jobs=n_jobs,
                    )
                    predicciones = modelo.predict(df_val[features])
                    deviance = mean_poisson_deviance(df_val["target_goles"], predicciones)
                    resultados.append({
                        "Liga": liga,
                        "Temporada_Validacion": temporada_val,
                        "Modelo": nombre_modelo,
                        "Estrategia": estrategia,
                        "Parametros": params,
                        "Deviance_CV_Interna": score_int,
                        "N_Folds_Internos": n_folds_int,
                        "Deviance": float(deviance),
                        "N_Train": len(df_train),
                        "N_Val": len(df_val),
                    })

    if not resultados:
        raise ValueError(
            "No se generaron folds externos. Necesitás más temporadas "
            "en desarrollo o reducir min_train_seasons."
        )

    detalle = pd.DataFrame(resultados).sort_values(
        ["Liga", "Temporada_Validacion", "Modelo", "Estrategia"]
    ).reset_index(drop=True)

    por_liga = (
        detalle.groupby(["Liga", "Modelo", "Estrategia"], as_index=False)
        .agg(
            Deviance_Promedio=("Deviance", "mean"),
            Deviance_STD=("Deviance", "std"),
            N_Temporadas=("Deviance", "size"),
        )
    )
    resumen = (
        por_liga.groupby(["Modelo", "Estrategia"], as_index=False)
        .agg(
            Deviance_Promedio_Ligas=("Deviance_Promedio", "mean"),
            Deviance_STD_Entre_Ligas=("Deviance_Promedio", "std"),
            N_Ligas=("Liga", "nunique"),
        )
    )
    comparacion = (
        resumen.pivot(index="Modelo", columns="Estrategia", values="Deviance_Promedio_Ligas")
        .rename(columns={"INTRA": "Deviance_INTRA", "GLOBAL": "Deviance_GLOBAL"})
        .reset_index()
    )
    if {"Deviance_INTRA", "Deviance_GLOBAL"}.issubset(comparacion.columns):
        comparacion["GLOBAL_menos_INTRA"] = (
            comparacion["Deviance_GLOBAL"] - comparacion["Deviance_INTRA"]
        )

    return {
        "detalle": detalle,
        "por_liga": por_liga,
        "resumen": resumen,
        "comparacion": comparacion,
        "test": test,
    }


In [ ]:
resultados = temporal_nested_cv(
    df=df,
    feature_sets=FEATURE_SETS,
    param_dict=PARAM_DICT,
    min_season=MIN_SEASON,
    max_season=MAX_SEASON,
    test_season=2025,
    min_train_seasons=MIN_TRAIN_SEASONS,
    min_train_seasons_inner=1,
    n_iter=N_ITER_RANDOM_SEARCH,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=True
)

Bundesliga | 2021 | INTRA | 1_Baseline_Localia
Bundesliga | 2021 | INTRA | 2_Elo
Bundesliga | 2021 | INTRA | 3_Prodegy_Puro
Bundesliga | 2021 | INTRA | 4_Prodegy_Con_Eff
Bundesliga | 2021 | INTRA | 5_Prodegy_Desbalance
Bundesliga | 2021 | INTRA | 6_Desbalances
Bundesliga | 2021 | INTRA | 7_Full_System
Bundesliga | 2021 | GLOBAL | 1_Baseline_Localia
Bundesliga | 2021 | GLOBAL | 2_Elo
Bundesliga | 2021 | GLOBAL | 3_Prodegy_Puro
Bundesliga | 2021 | GLOBAL | 4_Prodegy_Con_Eff
Bundesliga | 2021 | GLOBAL | 5_Prodegy_Desbalance
Bundesliga | 2021 | GLOBAL | 6_Desbalances
Bundesliga | 2021 | GLOBAL | 7_Full_System
Bundesliga | 2022 | INTRA | 1_Baseline_Localia
Bundesliga | 2022 | INTRA | 2_Elo
Bundesliga | 2022 | INTRA | 3_Prodegy_Puro
Bundesliga | 2022 | INTRA | 4_Prodegy_Con_Eff
Bundesliga | 2022 | INTRA | 5_Prodegy_Desbalance
Bundesliga | 2022 | INTRA | 6_Desbalances
Bundesliga | 2022 | INTRA | 7_Full_System
Bundesliga | 2022 | GLOBAL | 1_Baseline_Localia
Bundesliga | 2022 | GLOBAL | 2_Elo
B

In [ ]:
# Resultados de todos los folds externos
display(resultados["detalle"])

# Promedio por liga
display(resultados["por_liga"])

# Promedio general de los modelos
display(
    resultados["resumen"].sort_values(
        "Deviance_Promedio_Ligas"
    )
)

# Comparación INTRA vs GLOBAL
display(resultados["comparacion"])

,Liga,Temporada_Validacion,Modelo,Estrategia,Parametros,Deviance_CV_Interna,N_Folds_Internos,Deviance,N_Train,N_Val
0,Bundesliga,2021,1_Baseline_Localia,GLOBAL,{'modelo': 'PoissonRegressor'},NaN,0,1.303157,8062,612
1,Bundesliga,2021,1_Baseline_Localia,INTRA,{'modelo': 'PoissonRegressor'},NaN,0,1.282436,1224,612
2,Bundesliga,2021,2_Elo,GLOBAL,{'modelo': 'PoissonRegressor'},NaN,0,1.187325,8062,612
3,Bundesliga,2021,2_Elo,INTRA,{'modelo': 'PoissonRegressor'},NaN,0,1.162573,1224,612
4,Bundesliga,2021,3_Prodegy_Puro,GLOBAL,"{'max_iter': 300, 'max_depth': 3, 'learning_ra...",1.071237,1,1.145200,8062,612
...,...,...,...,...,...,...,...,...,...,...
331,Serie_A,2024,5_Prodegy_Desbalance,INTRA,{'modelo': 'PoissonRegressor'},NaN,0,1.170552,3800,760
332,Serie_A,2024,6_Desbalances,GLOBAL,"{'max_iter': 300, 'max_depth': 3, 'learning_ra...",1.063727,4,1.041848,20310,760
333,Serie_A,2024,6_Desbalances,INTRA,"{'max_iter': 300, 'max_depth': 3, 'learning_ra...",1.083163,4,1.042777,3800,760
334,Serie_A,2024,7_Full_System,GLOBAL,"{'max_iter': 300, 'max_depth': 3, 'learning_ra...",1.060770,4,1.038214,20310,760


,Liga,Modelo,Estrategia,Deviance_Promedio,Deviance_STD,N_Temporadas
0,Bundesliga,1_Baseline_Localia,GLOBAL,1.316018,0.027799,4
1,Bundesliga,1_Baseline_Localia,INTRA,1.290667,0.031791,4
2,Bundesliga,2_Elo,GLOBAL,1.207028,0.050127,4
3,Bundesliga,2_Elo,INTRA,1.179905,0.056678,4
4,Bundesliga,3_Prodegy_Puro,GLOBAL,1.156666,0.060760,4
...,...,...,...,...,...,...
79,Serie_A,5_Prodegy_Desbalance,INTRA,1.187661,0.012613,4
80,Serie_A,6_Desbalances,GLOBAL,1.057080,0.014038,4
81,Serie_A,6_Desbalances,INTRA,1.070709,0.022459,4
82,Serie_A,7_Full_System,GLOBAL,1.052394,0.014568,4


,Modelo,Estrategia,Deviance_Promedio_Ligas,Deviance_STD_Entre_Ligas,N_Ligas
12,7_Full_System,GLOBAL,1.105447,0.035844,6
6,4_Prodegy_Con_Eff,GLOBAL,1.107821,0.035031,6
10,6_Desbalances,GLOBAL,1.108879,0.035949,6
4,3_Prodegy_Puro,GLOBAL,1.109170,0.035037,6
5,3_Prodegy_Puro,INTRA,1.119339,0.031278,6
11,6_Desbalances,INTRA,1.120739,0.034657,6
7,4_Prodegy_Con_Eff,INTRA,1.121145,0.033913,6
13,7_Full_System,INTRA,1.126015,0.035206,6
3,2_Elo,INTRA,1.129093,0.036910,6
2,2_Elo,GLOBAL,1.132140,0.046271,6


Estrategia,Modelo,Deviance_GLOBAL,Deviance_INTRA,GLOBAL_menos_INTRA
0,1_Baseline_Localia,1.241426,1.238722,0.002703
1,2_Elo,1.132140,1.129093,0.003048
2,3_Prodegy_Puro,1.109170,1.119339,-0.010168
3,4_Prodegy_Con_Eff,1.107821,1.121145,-0.013324
4,5_Prodegy_Desbalance,1.227633,1.225040,0.002593
5,6_Desbalances,1.108879,1.120739,-0.011861
6,7_Full_System,1.105447,1.126015,-0.020568
